# CrossGatedHyena — Band Gap Prediction (Colab + GitHub Repo)

**Setup**: Clone your GitHub repo → download data from HuggingFace → train

All speed optimisations active:
- `CrossGatedHyenaDataset` (3.4× faster loading, no line-graph build)
- `EdgeBudgetBatchSampler` (capped VRAM, better GPU utilisation)
- Gradient checkpointing (~75% less activation VRAM, zero accuracy cost)
- AMP BF16/FP16 auto-selected by GPU capability
- `num_workers=4` + `persistent_workers` + `prefetch_factor=2`
- `torch.compile` **disabled** — 11× slower due to FFT graph-breaks


## 0 · Configuration — fill in before running

In [ ]:
# ── Repo ──────────────────────────────────────────────────────────────────
GITHUB_REPO   = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"  # ← fill in
REPO_NAME     = "YOUR_REPO"                                        # folder after clone
# If private repo, add token:  https://TOKEN@github.com/USER/REPO.git

# ── Data ──────────────────────────────────────────────────────────────────
HF_REPO_ID    = "YOUR_USERNAME/band-gap-prediction"  # ← fill in
DATA_DIR      = "/content/data"
CKPT_DIR      = "/content/checkpoints"   # download before session ends!
CKPT_NAME     = "crossgated_hyena_best.pt"

# ── Model ──────────────────────────────────────────────────────────────────
NODE_DIM      = 64
EDGE_DIM      = 64
NUM_LAYERS    = 4
NUM_RBF_POS   = 32
NUM_RBF_ANGLE = 16
MAX_K         = 96      # 95% of all graphs fully covered (from full-dataset scan)
FILTER_HIDDEN = 64
DROPOUT       = 0.1

# ── Training ───────────────────────────────────────────────────────────────
EPOCHS        = 100
EDGE_BUDGET   = 25_000  # max edges per batch (replaces fixed batch_size)
LR            = 1e-3
WEIGHT_DECAY  = 1e-5
NUM_WORKERS   = 4
MAX_IDS       = None    # None = full ~100K; set e.g. 2000 for a quick test
ACCUM_STEPS   = 4       # gradient accumulation → effective batch ≈ budget × 4
RESUME        = True    # auto-resume from checkpoint if it exists


## 1 · Clone repo

In [ ]:
import os, sys

REPO_DIR = f"/content/{REPO_NAME}"

if os.path.exists(REPO_DIR):
    print(f"Repo already at {REPO_DIR} — pulling latest …")
    os.system(f"git -C {REPO_DIR} pull")
else:
    print(f"Cloning {GITHUB_REPO} …")
    ret = os.system(f"git clone {GITHUB_REPO} {REPO_DIR}")
    if ret != 0:
        raise RuntimeError("git clone failed — check GITHUB_REPO and token")

# Add repo root to Python path so 'from models.geometric_hyena import …' works
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Quick sanity check
for rel in ["utils/graph_utils.py", "data/cross_gated_hyena_dataset.py",
            "models/geometric_hyena.py"]:
    path = os.path.join(REPO_DIR, rel)
    status = "✓" if os.path.exists(path) else "✗ MISSING"
    print(f"  {status}  {rel}")


## 2 · Install dependencies

In [ ]:
import subprocess, sys, torch

print(f"PyTorch {torch.__version__}  |  CUDA {torch.version.cuda}")

# PyTorch Geometric — must match torch+cuda version
pyg_url = (
    f"https://data.pyg.org/whl/torch-"
    f"{torch.__version__.split('+')[0]}+cu{torch.version.cuda.replace('.','')}.html"
)
subprocess.check_call([sys.executable, "-m", "pip", "install",
    "torch_geometric", "torch_scatter", "torch_sparse",
    "-f", pyg_url, "-q"])

subprocess.check_call([sys.executable, "-m", "pip",
    "install", "h5py", "huggingface_hub", "tqdm", "-q"])

print("All packages installed.")


## 3 · GPU + AMP dtype

In [ ]:
import torch, os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if device.type == "cuda":
    p   = torch.cuda.get_device_properties(0)
    cap = (p.major, p.minor)
    print(f"GPU  : {p.name}  (compute {cap[0]}.{cap[1]})")
    print(f"VRAM : {p.total_memory / 1e9:.1f} GB")

    # BF16 Tensor Cores: Ampere (8.0+). T4 = 7.5 → FP16 + GradScaler
    AMP_DTYPE       = torch.bfloat16 if cap[0] >= 8 else torch.float16
    USE_GRAD_SCALER = (AMP_DTYPE == torch.float16)
    USE_AMP         = True

    # TF32: free ~10% throughput on matmuls (Ampere+)
    torch.backends.cuda.matmul.fp32_precision = "tf32"
    torch.backends.cudnn.conv.fp32_precision  = "tf32"
    print(f"AMP  : {AMP_DTYPE}  |  GradScaler : {USE_GRAD_SCALER}")
    print("TF32 : enabled")
else:
    AMP_DTYPE = torch.float32; USE_AMP = False; USE_GRAD_SCALER = False
    print("WARNING: No GPU — training will be very slow.")

print(f"Device : {device}")


## 4 · Download data from HuggingFace

> Upload your files first if you haven't already:
> ```bash
> huggingface-cli upload YOUR_USERNAME/band-gap-prediction graphs_data.h5 --repo-type dataset
> huggingface-cli upload YOUR_USERNAME/band-gap-prediction materials_tabular.csv --repo-type dataset
> ```


In [ ]:
from huggingface_hub import hf_hub_download
import shutil, os

os.makedirs(DATA_DIR,  exist_ok=True)
os.makedirs(CKPT_DIR,  exist_ok=True)

H5_PATH  = os.path.join(DATA_DIR, "graphs_data.h5")
CSV_PATH = os.path.join(DATA_DIR, "materials_tabular.csv")

for fname, local in [("graphs_data.h5", H5_PATH), ("materials_tabular.csv", CSV_PATH)]:
    if os.path.exists(local):
        print(f"{fname}: already present ({os.path.getsize(local)/1e9:.2f} GB)")
        continue
    print(f"Downloading {fname} …")
    tmp = hf_hub_download(repo_id=HF_REPO_ID, filename=fname, repo_type="dataset")
    shutil.move(tmp, local)
    print(f"  saved → {local}  ({os.path.getsize(local)/1e9:.2f} GB)")

print("\nData ready.")


## 5 · Import from cloned repo

In [ ]:
# All imports come from the cloned repo — no %%writefile needed
from data.cross_gated_hyena_dataset import CrossGatedHyenaDataset, EdgeBudgetBatchSampler
from models.geometric_hyena import CrossGatedHyena
from torch_geometric.loader import DataLoader
from torch.utils.data import random_split
import torch

print("Imports OK:")
print(f"  CrossGatedHyenaDataset  from {CrossGatedHyenaDataset.__module__}")
print(f"  CrossGatedHyena         from {CrossGatedHyena.__module__}")


## 6 · Dataset + `EdgeBudgetBatchSampler`

- `CrossGatedHyenaDataset`: 3.4× faster than ALIGNNDataset — no line-graph build
- `EdgeBudgetBatchSampler`: each batch ≤ `EDGE_BUDGET` total edges  
  → no OOM from variable-size graphs; small crystals batch together efficiently


In [ ]:
print("Indexing dataset …")
full_ds = CrossGatedHyenaDataset(H5_PATH, CSV_PATH, num_rbf=NUM_RBF_POS, max_ids=MAX_IDS)
print(f"Total  : {len(full_ds):,} materials")

n = len(full_ds)
n_tr = int(0.80*n); n_val = int(0.10*n); n_te = n - n_tr - n_val
gen  = torch.Generator().manual_seed(42)
tr_ds, val_ds, te_ds = random_split(full_ds, [n_tr, n_val, n_te], generator=gen)
print(f"Train {n_tr:,}  Val {n_val:,}  Test {n_te:,}")

# Edge-budget samplers: one-time H5 metadata read to cache edge counts
tr_samp  = EdgeBudgetBatchSampler(tr_ds,  EDGE_BUDGET, shuffle=True)
val_samp = EdgeBudgetBatchSampler(val_ds, EDGE_BUDGET, shuffle=False)
te_samp  = EdgeBudgetBatchSampler(te_ds,  EDGE_BUDGET, shuffle=False)
print(f"Train batches : {len(tr_samp):,}  (EDGE_BUDGET={EDGE_BUDGET:,})")

_ldr = dict(num_workers=NUM_WORKERS, pin_memory=True,
            persistent_workers=(NUM_WORKERS > 0),
            prefetch_factor=2 if NUM_WORKERS > 0 else None)
train_loader = DataLoader(tr_ds,  batch_sampler=tr_samp,  **_ldr)
val_loader   = DataLoader(val_ds, batch_sampler=val_samp, **_ldr)
test_loader  = DataLoader(te_ds,  batch_sampler=te_samp,  **_ldr)

b = next(iter(train_loader))
print(f"\nSample batch:")
print(f"  x_cat        : {b.x_cat.shape}    x : {b.x.shape}")
print(f"  edge_cat     : {b.edge_cat.shape}  edge_attr : {b.edge_attr.shape}")
print(f"  y (band gap) : {b.y.shape}  range [{b.y.min():.2f}, {b.y.max():.2f}] eV")


## 7 · Model

In [ ]:
model = CrossGatedHyena(
    node_in_dim            = 8,           # x_lin(4 active) + x_log(4)
    edge_in_dim            = NUM_RBF_POS, # RBF-only distance features
    node_dim               = NODE_DIM,
    edge_dim               = EDGE_DIM,
    num_layers             = NUM_LAYERS,
    num_rbf_pos            = NUM_RBF_POS,
    num_rbf_angle          = NUM_RBF_ANGLE,
    max_k                  = MAX_K,       # 96 → 95% of graphs fully covered
    filter_hidden          = FILTER_HIDDEN,
    dropout                = DROPOUT,
    gradient_checkpointing = True,        # ~75% less activation VRAM
).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"CrossGatedHyena — {n_params:,} parameters")
print(f"  max_k={MAX_K}  |  grad_ckpt=True  |  AMP={AMP_DTYPE}")

# Sanity check
model.eval()
with torch.no_grad():
    with torch.autocast(device.type, dtype=AMP_DTYPE, enabled=USE_AMP):
        out = model(b.to(device))
print(f"\nForward check: {tuple(out.shape)}  ✓")


## 8 · Training

In [ ]:
import time, torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm.notebook import tqdm

criterion = nn.HuberLoss(delta=0.5)
l1_loss   = nn.L1Loss()
optim     = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optim, T_max=EPOCHS, eta_min=LR * 1e-3)
scaler    = torch.amp.GradScaler(enabled=USE_GRAD_SCALER)

ckpt_path    = os.path.join(CKPT_DIR, CKPT_NAME)
best_val_mae = float("inf")
start_epoch  = 1
history      = {"loss": [], "mae": [], "val_mae": [], "val_rmse": []}

# Auto-resume
if RESUME and os.path.exists(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model"])
    optim.load_state_dict(ckpt["optim"])
    scheduler.load_state_dict(ckpt["scheduler"])
    scaler.load_state_dict(ckpt["scaler"])
    best_val_mae = ckpt["val_mae"]
    start_epoch  = ckpt["epoch"] + 1
    history      = ckpt.get("history", history)
    print(f"Resumed from epoch {ckpt['epoch']}  (val MAE = {best_val_mae:.4f} eV)")
else:
    print(f"Fresh training — epochs 1..{EPOCHS}")


@torch.no_grad()
def evaluate(loader, desc="eval"):
    model.eval()
    mae_sum = mse_sum = n = 0
    for b in tqdm(loader, desc=desc, leave=False):
        b = b.to(device, non_blocking=True)
        with torch.autocast(device.type, dtype=AMP_DTYPE, enabled=USE_AMP):
            pred = model(b).squeeze().float()
        tgt      = b.y.squeeze().float()
        mae_sum += (pred - tgt).abs().sum().item()
        mse_sum += ((pred - tgt) ** 2).sum().item()
        n       += tgt.numel()
    return mae_sum / n, (mse_sum / n) ** 0.5


print(f"Accum steps : {ACCUM_STEPS}  (effective budget × {ACCUM_STEPS})\n")

for epoch in range(start_epoch, EPOCHS + 1):
    tr_samp.set_epoch(epoch)   # re-shuffle batch groupings each epoch

    model.train()
    t0 = time.perf_counter()
    total_loss = total_mae = n_step = 0
    optim.zero_grad(set_to_none=True)

    bar = tqdm(train_loader, desc=f"Ep {epoch:3d}/{EPOCHS}", leave=True)
    for step, b in enumerate(bar):
        b = b.to(device, non_blocking=True)

        # ── Single forward pass — derive all metrics from one prediction ──
        with torch.autocast(device.type, dtype=AMP_DTYPE, enabled=USE_AMP):
            pred = model(b).squeeze()
            tgt  = b.y.squeeze().float()
            loss = criterion(pred, tgt) / ACCUM_STEPS

        scaler.scale(loss).backward()

        with torch.no_grad():
            p_f         = pred.detach().float()
            total_loss += loss.item() * ACCUM_STEPS
            total_mae  += (p_f - tgt).abs().mean().item()
            n_step     += 1

        if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == len(train_loader):
            scaler.unscale_(optim)
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(optim); scaler.update()
            optim.zero_grad(set_to_none=True)

        bar.set_postfix({
            "l":   f"{total_loss/n_step:.3f}",
            "mae": f"{total_mae/n_step:.3f}",
        })

    scheduler.step()
    val_mae, val_rmse = evaluate(val_loader, "  val")
    torch.cuda.empty_cache()

    history["loss"].append(total_loss / max(n_step, 1))
    history["mae"].append(total_mae / max(n_step, 1))
    history["val_mae"].append(val_mae)
    history["val_rmse"].append(val_rmse)

    star = ""
    if val_mae < best_val_mae:
        best_val_mae = val_mae
        torch.save({
            "epoch"     : epoch,
            "model"     : model.state_dict(),
            "optim"     : optim.state_dict(),
            "scheduler" : scheduler.state_dict(),
            "scaler"    : scaler.state_dict(),
            "val_mae"   : val_mae,
            "val_rmse"  : val_rmse,
            "history"   : history,
            "config"    : {"node_dim": NODE_DIM, "edge_dim": EDGE_DIM,
                           "num_layers": NUM_LAYERS, "max_k": MAX_K},
        }, ckpt_path)
        star = "  ★"

    t = time.perf_counter() - t0
    print(f"  loss={history['loss'][-1]:.4f}  train_MAE={history['mae'][-1]:.4f}  "
          f"val_MAE={val_mae:.4f}  val_RMSE={val_rmse:.4f}  "
          f"lr={scheduler.get_last_lr()[0]:.1e}  {t:.0f}s{star}")

print(f"\nDone.  Best val MAE = {best_val_mae:.4f} eV")
print(f"Checkpoint: {ckpt_path}  ← download before session ends!")


## 9 · Test evaluation

In [ ]:
ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
model.load_state_dict(ckpt["model"])
print(f"Loaded epoch {ckpt['epoch']}  val MAE = {ckpt['val_mae']:.4f} eV")

test_mae, test_rmse = evaluate(test_loader, "test")
print(f"\nTest  MAE  : {test_mae:.4f} eV")
print(f"Test  RMSE : {test_rmse:.4f} eV")


## 10 · Training curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle(f"CrossGatedHyena  max_k={MAX_K}  layers={NUM_LAYERS}  dim={NODE_DIM}", fontsize=12)

axes[0].plot(history["loss"], label="Huber loss", color="tab:blue")
axes[0].plot(history["mae"],  label="Train MAE",  color="tab:green", linestyle="--")
axes[0].set(title="Training", xlabel="Epoch", ylabel="Value")
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(history["val_mae"],  label="Val MAE",  color="tab:orange")
axes[1].plot(history["val_rmse"], label="Val RMSE", color="tab:red", linestyle="--")
axes[1].axhline(test_mae,  color="tab:orange", linestyle=":",
                label=f"Test MAE  {test_mae:.3f} eV", alpha=0.8)
axes[1].axhline(test_rmse, color="tab:red",    linestyle=":",
                label=f"Test RMSE {test_rmse:.3f} eV", alpha=0.8)
axes[1].set(title="Validation / Test", xlabel="Epoch", ylabel="eV")
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("/content/training_curves.png", dpi=150)
plt.show()
print("Saved → /content/training_curves.png")


## 11 · Download checkpoint before session ends

In [ ]:
from google.colab import files
import os

if os.path.exists(ckpt_path):
    print(f"Downloading {CKPT_NAME} ({os.path.getsize(ckpt_path)/1e6:.1f} MB) …")
    files.download(ckpt_path)
else:
    print("No checkpoint found — has training completed?")
